# Data Preparation — CUB-200-2011 Tabular Attributes

This notebook turns the raw CUB-200-2011 attribute files into tidy feature tables (one row per image) for the classification models (Decision Tree / Random Forest, KNN, Logistic Regression).

Findings and the justification for each step are in [`tabularDataAnalysis.md`](../tabularDataAnalysis.md).

**Outputs** (written to `data/`):

| File | Features | Not-visible handling |
|---|---|---|
| `cub_features_A.csv` | 312 binary attributes | treated as absent (0), baseline |
| `cub_features_B.csv` | 312 attributes in {−1, 0, 1} | encoded as −1 |
| `cub_features_C.csv` | 312 binary + 28 `notvis::<group>` flags | separate per-body-part flag |

Every file also contains `image_id`, `class_id`, `class_name` and two train/test split columns:
- `is_train`: the official dataset split (about 50/50);
- `is_train_80`: a stratified 80/20 split (see §8).

**Excluded on purpose:** `parts/` and `bounding_boxes.txt` (pixel-level image annotations), `class_attribute_labels_continuous.txt` (computed from all images, including test, so using it would leak the target), and the `time` column (describes the annotator, not the bird).

## 1. Setup

- Paths are resolved relative to the repo root, so the notebook works whether Jupyter is started from the root or from `notebooks/`.
- `DROP_GUESSING_POSITIVES` is an optional cleaning switch (see §6). It is **false by default** meaning any guessed attributes are left as present. If set to true, guessed attributes will be treated as absent.
- All CUB files are whitespace-separated with no header, so a single `read_ws` helper is used everywhere. Splitting on any whitespace also strips the Windows line endings (`\r\n`) in the clean attribute file.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / 'data/raw/CUB_200_2011').exists() else Path.cwd().parent
CUB = ROOT / 'data/raw/CUB_200_2011'
OUT = ROOT / 'data'
OUT.mkdir(exist_ok=True)

# Set to True to treat attributes the annotator marked as "guessing" as absent (see section 6)
DROP_GUESSING_POSITIVES = False

# Alternative stratified train/test split (see section 8)
TEST_SIZE_80_20 = 0.2
SPLIT_SEED = 42

def read_ws(path, names):
    """Read a whitespace-separated, headerless CUB file (also strips CRLF line endings)."""
    return pd.read_csv(path, sep=r'\s+', header=None, names=names)

## 2. Load the metadata files

Only the files needed for tabular classification are loaded:
- the species names and image list;
- the label for each image;
- the official train/test split;
- the attribute names;
- the certainty scale.

The image-derived files are deliberately left out (see the note at the top). Each attribute name has the form `group::value`, so it is split into its body-part group (28 groups) and value. This lets later steps work per body part, both for the "not visible" flags and for grouped feature importance.

In [3]:
classes = read_ws(CUB / 'classes.txt', ['class_id', 'class_name'])
images = read_ws(CUB / 'images.txt', ['image_id', 'path'])
labels = read_ws(CUB / 'image_class_labels.txt', ['image_id', 'class_id'])
split = read_ws(CUB / 'train_test_split.txt', ['image_id', 'is_train'])
# Certainty names contain spaces ("not visible"), so split on the first space only
with open(CUB / 'attributes' / 'certainties.txt') as f:
    certainties = pd.DataFrame([line.strip().split(' ', 1) for line in f if line.strip()],
                               columns=['certainty_id', 'certainty_name'])
certainties['certainty_id'] = certainties.certainty_id.astype(int)

attributes = pd.read_csv(CUB / 'attributes' / 'attributes.txt', sep=' ', header=None,
                         names=['attr_id', 'attr_name'])
attributes[['group', 'value']] = attributes['attr_name'].str.split('::', expand=True)

print(f'{len(classes)} classes, {len(images)} images, {len(attributes)} attributes '
      f'in {attributes.group.nunique()} groups')
certainties
attributes

200 classes, 11788 images, 312 attributes in 28 groups


,attr_id,attr_name,group,value
0,1,has_bill_shape::curved_(up_or_down),has_bill_shape,curved_(up_or_down)
1,2,has_bill_shape::dagger,has_bill_shape,dagger
2,3,has_bill_shape::hooked,has_bill_shape,hooked
3,4,has_bill_shape::needle,has_bill_shape,needle
4,5,has_bill_shape::hooked_seabird,has_bill_shape,hooked_seabird
...,...,...,...,...
307,308,has_crown_color::buff,has_crown_color,buff
308,309,has_wing_pattern::solid,has_wing_pattern,solid
309,310,has_wing_pattern::spotted,has_wing_pattern,spotted
310,311,has_wing_pattern::striped,has_wing_pattern,striped


## 3. Load the image attribute labels

We use `image_attribute_labels_clean.txt` (shipped with the Kaggle copy of the dataset). The original `image_attribute_labels.txt` has 606 malformed rows for images 2275 and 9364, which contain a stray extra `0` field. The clean file is the original with that field removed. The next cell checks this claim, so it doesn't have to be taken on trust.

In [4]:
COLS = ['image_id', 'attr_id', 'is_present', 'certainty', 'time']
ial = read_ws(CUB / 'attributes' / 'image_attribute_labels_clean.txt', COLS)
ial[ial['image_id']==2275].head()

,image_id,attr_id,is_present,certainty,time
709488,2275,1,0,4,1025.283
709489,2275,2,0,4,1025.283
709490,2275,3,0,4,1025.283
709491,2275,4,0,4,1025.283
709492,2275,5,0,4,1025.283


In [5]:
# Verify the clean file against the raw file: parse the raw file line by line,
# drop the stray 5th field on 6-field rows, and compare.
with open(CUB / 'attributes' / 'image_attribute_labels.txt') as f:
    rows = [line.split() for line in f]
bad = [r for r in rows if len(r) == 6]
fixed = [r[:4] + r[5:] if len(r) == 6 else r for r in rows]
raw_fixed = pd.DataFrame(fixed, columns=COLS).astype(ial.dtypes.to_dict())

print('malformed raw rows:', len(bad), '| affected images:', sorted({int(r[0]) for r in bad}))
print('clean file == repaired raw file:', raw_fixed.equals(ial))
del rows, fixed, raw_fixed

malformed raw rows: 606 | affected images: [2275, 9364]
clean file == repaired raw file: True


## 4. Integrity checks

These checks are written as `assert`s, so the notebook **stops** if the data is not as expected, instead of silently producing wrong features. Each check rules out a specific way the pipeline could go wrong:

| Check | Why it matters |
|---|---|
| No missing values | Nothing needs imputing, so no imputation strategy has to be justified |
| Exactly one row per (image, attribute) | The long-to-wide pivot (§5) is valid and loses nothing |
| Same image IDs in every file | Joins on `image_id` don't drop or duplicate images |
| Folder prefix matches the class label | The labels file and image folders agree (guards against an off-by-one class ID) |
| "Not visible" always has `is_present = 0` | Confirms that a 0 can mean *absent* **or** *hidden*. This is the problem the variants in §7 address |
| Certainty is constant within a body-part group | One certainty answer is given per body part, so a single flag per group (variant C) captures all of the visibility information |

In [6]:
n_img, n_attr = len(images), len(attributes)

assert ial.isna().sum().sum() == 0, 'missing values in attribute labels'
assert len(ial) == n_img * n_attr, 'expected one row per (image, attribute)'
assert not ial.duplicated(['image_id', 'attr_id']).any(), 'duplicate (image, attribute) rows'
assert set(ial.image_id) == set(images.image_id) == set(labels.image_id) == set(split.image_id)
assert set(ial.is_present) == {0, 1}
assert set(ial.certainty) == set(certainties.certainty_id)

# The class prefix in each image's folder name must match its label
prefix = images.set_index('image_id').path.str[:3].astype(int)
assert (prefix == labels.set_index('image_id').class_id).all()

# "not visible" (certainty 1) is always recorded as is_present = 0
assert ial.loc[ial.certainty == 1, 'is_present'].eq(0).all()

# Certainty is given once per body part (attribute group), not per attribute
grp = ial.attr_id.map(attributes.set_index('attr_id').group)
assert ial.groupby([ial.image_id, grp]).certainty.nunique().eq(1).all()

print('All integrity checks passed.')

All integrity checks passed.


## 5. Reshape from long to wide

The label file has one row per (image, attribute). We pivot it into an 11,788 × 312 matrix and use readable column names (`has_wing_color::blue`, …) so tree splits can be interpreted directly.

**Why keep 312 binary columns instead of 28 categorical ones?** Colour groups are multi-label: an image can have up to 4 colours ticked for the same body part (for example, 57% of images have more than one wing colour). Collapsing each group into a single categorical column would discard that information. Binary columns are also the natural input for trees, which split on one column at a time. The certainty matrix is pivoted too, because §6 and §7 need it.

In [7]:
attr_names = attributes.set_index('attr_id').attr_name

present = ial.pivot(index='image_id', columns='attr_id', values='is_present').rename(columns=attr_names)
certainty = ial.pivot(index='image_id', columns='attr_id', values='certainty').rename(columns=attr_names)
present.columns.name = certainty.columns.name = None

print(present.shape)
present.iloc[:5, :8]

(11788, 312)


,has_bill_shape::curved_(up_or_down),has_bill_shape::dagger,has_bill_shape::hooked,has_bill_shape::needle,has_bill_shape::hooked_seabird,has_bill_shape::spatulate,has_bill_shape::all-purpose,has_bill_shape::cone
image_id,,,,,,,,
1,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,0
5,0,0,0,0,1,0,0,0


## 6. Optional: use annotator certainty

When `DROP_GUESSING_POSITIVES` is `True`, attributes the worker marked present while only *guessing* (certainty 2, about 27k cells, 7% of all positives) are set to 0. The idea is that guesses are the noisiest labels, so removing them might reduce noise, at the cost of losing some correct positives.

**Status: not evaluated.** The model notebooks were run with this switch off, so its effect is unknown. Testing it means setting it to `True`, saving the output under a new name, and running the same CV comparison as for variants A/B/C. This is listed as future work.

In [8]:
guessing_pos = (certainty == 2) & (present == 1)
print(f'positives marked "guessing": {int(guessing_pos.values.sum()):,} '
      f'of {int(present.values.sum()):,} total positives')

if DROP_GUESSING_POSITIVES:
    present = present.mask(guessing_pos, 0)
    print('-> guessing positives set to 0')

positives marked "guessing": 27,306 of 371,047 total positives


## 7. Build the three "not visible" variants

When a body part is hidden, the worker chose certainty 1 ("not visible") and every attribute in that group is recorded as 0, which is indistinguishable from "absent". We build three encodings and compare them later with cross-validation:

- **A**: raw 0/1 (baseline, "not visible" = absent)
- **B**: 1 = present, 0 = absent, −1 = not visible
- **C**: A plus one `notvis::<group>` flag per body part (28 columns)

Certainty is constant within a group (checked above), so the per-group flags in C keep all the information.

In [10]:
not_visible = certainty == 1
group_of = attributes.set_index('attr_name').group

X_A = present.copy()

X_B = present.mask(not_visible, -1)

notvis_groups = not_visible.T.groupby(group_of).any().T.astype(int)
notvis_groups.columns = 'notvis::' + notvis_groups.columns
X_C = pd.concat([present, notvis_groups], axis=1)

for name, X in [('A', X_A), ('B', X_B), ('C', X_C)]:
    print(name, X.shape, 'values:', sorted(np.unique(X.values)))

A (11788, 312) values: [np.int64(0), np.int64(1)]
B (11788, 312) values: [np.int64(-1), np.int64(0), np.int64(1)]
C (11788, 340) values: [np.int64(0), np.int64(1)]


## 8. Attach labels and the official train/test split

We use the dataset's **official split** (5,994 train / 5,794 test). There are two reasons:
- results stay comparable with other work on CUB-200-2011;
- it already gives **29–30 training images for every species**.

Because of the second point, the lecturer's suggestion to subsample every class down to the smallest one (41 images) is unnecessary: the training set is already balanced. The test set is less balanced (11–30 images per species), which is why the model notebooks report **macro-F1** alongside accuracy.

### Alternative: stratified 80/20 split (`is_train_80`)

A second split column is added so that models can also be trained on more data. It is a new column rather than a replacement: the official split stays available, and notebooks already using `is_train` keep working unchanged.

- **Stratified by species** with a fixed seed (`SPLIT_SEED = 42`). Every species keeps 80% of its images for training (33–48 each) and 20% for testing (8–12 each, 2,358 test images in total). Everyone who runs this notebook gets the identical split.
- **No fixed validation set.** Tuning should use stratified 5-fold CV *within* the 80% training portion (about 7–10 images per species per fold). That uses the data more efficiently than holding out another 10% (about 6 images per species), which would give a noisy score and take away training data, the main limiting factor.
- **Why consider it:** a scratch experiment with the tuned forest gave about 48% accuracy on 80/20 splits, against 45.6% on the official split. Capping 80/20 training at 30 images per species brought it back to about 45.7%, so the gain comes from the extra training images, not from an easier test set.
- **Trade-offs:**
  - the test set is smaller, so accuracy is noisier (about ±1.0 vs ±0.65 points) and per-species F1 on 8–12 images is unreliable;
  - the training classes are slightly uneven (33–48), which macro-F1 and `class_weight` can account for;
  - results are no longer comparable with published work, which uses the official split.
- **All models being compared must use the same split column.** That is a team decision.

In [11]:
meta = (labels.merge(classes, on='class_id')
              .merge(split, on='image_id')
              .set_index('image_id')
              .sort_index())

# Stratified 80/20 split: same species proportions in train and test, reproducible via SPLIT_SEED
train_ids, _ = train_test_split(meta.index, test_size=TEST_SIZE_80_20,
                                stratify=meta.class_id, random_state=SPLIT_SEED)
meta['is_train_80'] = meta.index.isin(train_ids).astype(int)

def with_meta(X):
    return X.join(meta, how='left').reset_index()

df_A, df_B, df_C = with_meta(X_A), with_meta(X_B), with_meta(X_C)
assert df_A[['class_id', 'is_train', 'is_train_80']].notna().all().all()
df_A[['image_id', 'class_id', 'class_name', 'is_train', 'is_train_80']].head()

,image_id,class_id,class_name,is_train,is_train_80
0,1,1,001.Black_footed_Albatross,0,1
1,2,1,001.Black_footed_Albatross,1,1
2,3,1,001.Black_footed_Albatross,0,1
3,4,1,001.Black_footed_Albatross,1,1
4,5,1,001.Black_footed_Albatross,1,1


## 9. Sanity summaries

A final look at the prepared data before saving, to confirm the decisions above:
- **Class balance:** the official training split should be flat at 29–30 images per class. This confirms that no resampling or class weighting is needed for it. The 80/20 split mirrors each species' total (41–60 images), so its training counts vary between 33 and 48.
- **Attribute prevalence:** most attributes are rare (median about 4% of images, 27 below 0.5%). Rare attributes are **kept**. Dropping them would be an untested choice, and trees simply never split on uninformative columns, so keeping them costs nothing. The tree and forest feature importances later show which attributes actually matter.
- **Not-visible rate per body part:** shows how much of the data variants B and C actually change. Wing shape (35%), upper-tail colour (33%) and the other tail and back groups (22–26%) are hidden most often. The bill and head are hidden in under 5% of images. Size, shape and primary colour are never marked not visible.

**No scaling or normalisation is applied.** Trees are invariant to monotonic feature transforms, and the features are already on the same 0/1 (or −1/0/1) scale. Distance- and gradient-based models (KNN, Logistic Regression) should decide on scaling in their own notebooks.

In [ ]:
for split_col in ['is_train', 'is_train_80']:
    counts = meta.groupby(['class_id', split_col]).size().unstack().rename(columns={0: 'test', 1: 'train'})
    print(f'{split_col}: train / test images = {int(meta[split_col].sum())} / {int((meta[split_col] == 0).sum())}, '
          f'per class train {counts.train.min()}-{counts.train.max()}, test {counts.test.min()}-{counts.test.max()}')

# The 80/20 split must keep each species' 80/20 proportion (stratification check)
share = meta.groupby('class_id').is_train_80.mean()
assert share.between(0.75, 0.85).all(), share.describe()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for split_col, style in [('is_train', '-'), ('is_train_80', '--')]:
    counts = meta.groupby(['class_id', split_col]).size().unstack().rename(columns={0: 'test', 1: 'train'})
    counts.train.plot(ax=axes[0], lw=1, ls=style, label=f'{split_col} train')
    counts.test.plot(ax=axes[0], lw=1, ls=style, label=f'{split_col} test')
axes[0].set(title='Images per class', xlabel='class_id', ylabel='images')
axes[0].legend(fontsize=7)

X_A.mean().sort_values().reset_index(drop=True).plot(ax=axes[1], lw=1.5)
axes[1].set(title='Attribute prevalence (sorted)', xlabel='attribute rank', ylabel='share of images present')

notvis_groups.mean().sort_values().plot.barh(ax=axes[2], fontsize=6)
axes[2].set(title='Share of images where body part is not visible')
plt.tight_layout()
plt.show()

In [ ]:
dup = X_A.duplicated(keep=False)
conflicts = (X_A[dup].join(meta.class_id)
             .groupby(list(X_A.columns)).class_id.nunique().gt(1).sum())
print(f'images sharing an identical feature vector: {int(dup.sum())} '
      f'({conflicts} groups with conflicting species, irreducible error)')
print(f'rare attributes (<0.5% of images): {(X_A.mean() < 0.005).sum()}  (kept; trees ignore them)')

images sharing an identical feature vector: 55 (16 groups with conflicting species, irreducible error)
rare attributes (<0.5% of images): 27  (kept; trees ignore them)


**Duplicates:** images with identical feature vectors are kept. They are genuine, separate photos that happen to receive the same annotation, so removing them would be arbitrary. Where identical vectors carry **different** species (16 groups), no model can predict both correctly. This sets a small, unavoidable ceiling on accuracy and is noted as a limitation.

## 10. Save

Each variant is saved as one flat CSV containing metadata plus features. This lets every model notebook (DT/RF, KNN, LogReg) start from identical, already-validated data with a single `read_csv`. The CSVs are git-ignored because they can always be regenerated by re-running this notebook.

Model notebooks load one of these files, then split on **one** of the split columns, `is_train` (official) or `is_train_80` (1 = train, 0 = test). Both split columns must be excluded from the features. Hyperparameters should be tuned with stratified cross-validation on the training rows only.

In [12]:
for name, df in [('A', df_A), ('B', df_B), ('C', df_C)]:
    path = OUT / f'cub_features_{name}.csv'
    df.to_csv(path, index=False)
    print(f'{path.relative_to(ROOT)}: {df.shape}')

data\cub_features_A.csv: (11788, 317)
data\cub_features_B.csv: (11788, 317)
data\cub_features_C.csv: (11788, 345)


In [ ]:
# Example of loading in a model notebook
SPLIT = 'is_train_80'  # or 'is_train' for the official split
META_COLS = ['image_id', 'class_id', 'class_name', 'is_train', 'is_train_80']

df = pd.read_csv(OUT / 'cub_features_A.csv')
feature_cols = [c for c in df.columns if c not in META_COLS]
train, test = df[df[SPLIT] == 1], df[df[SPLIT] == 0]
X_train, y_train = train[feature_cols], train.class_id
X_test, y_test = test[feature_cols], test.class_id
print(X_train.shape, X_test.shape)

(9430, 312) (2358, 312)
